# 01 — Offline assets

Run once on the developer machine after a fresh GLB batch arrives.
Generates the derived files the viewer consumes:

1. `public/manifest.json` — array form sorted by `stem`, parsed from `public/manifest.jsonl`.
2. `public/thumbnails/<stem>.webp` — 256x256, fixed camera, transparent background.
3. `public/stats.json` — count and basic stats from the manifest.
4. Validation: every GLB parses with `pygltflib`.
5. Optional: batch compress with `gltf-transform` (off by default).

Dependencies: `pyrender`, `trimesh`, `numpy`, `pillow`, `pygltflib`.

In [1]:
!pip install --quiet "pyglet<2" pillow


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
!pip install pyrender trimesh numpy pillow pygltflib


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import os, sys, json, math, statistics, subprocess
from pathlib import Path

ROOT = Path('..').resolve()
PUBLIC = ROOT / 'public'
MODELS = PUBLIC / 'models'
THUMBS = PUBLIC / 'thumbnails'
JSONL = PUBLIC / 'manifest.jsonl'
MANIFEST = PUBLIC / 'manifest.json'
STATS = PUBLIC / 'stats.json'

THUMBS.mkdir(parents=True, exist_ok=True)
print('root:', ROOT)
print('models:', MODELS, '->', len(list(MODELS.glob('*.glb'))), 'glb files')

root: C:\Users\Syauqi Nabil\Project\ARCA
models: C:\Users\Syauqi Nabil\Project\ARCA\public\models -> 60 glb files


## 1. Parse manifest.jsonl into manifest.json

In [2]:
raw = []
with open(JSONL, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        raw.append(json.loads(line))

# Dedupe by stem, keep the last occurrence so re-runs of the J1 pipeline
# overwrite older entries.
by_stem = {}
for entry in raw:
    by_stem[entry['stem']] = entry
entries = sorted(by_stem.values(), key=lambda e: e['stem'])

with open(MANIFEST, 'w', encoding='utf-8') as f:
    json.dump(entries, f, indent=2)

print(f'read {len(raw)} lines, wrote {MANIFEST} with {len(entries)} unique entries')

read 65 lines, wrote C:\Users\Syauqi Nabil\Project\ARCA\public\manifest.json with 60 unique entries


## 2. Render thumbnails

Uses `trimesh.Scene.save_image` with pyglet for a fixed-camera thumbnail per GLB.
Camera orbits the bounding sphere at 45 degrees yaw and 20 degrees pitch.
Output is 256x256 WebP on the paper background.

Pyrender + PyOpenGL is avoided because its ctypes array marshalling crashes
on Windows with current numpy. Pyglet ships its own GL bindings and works.

In [3]:
import io
import numpy as np
import trimesh
from PIL import Image

TH_SIZE = 256
YAW_DEG = 45.0
PITCH_DEG = 20.0
FOV_DEG = 35.0
FILL = 0.7
PAPER = (247, 245, 240, 255)  # matches --paper

def camera_transform(radius):
    yaw = math.radians(YAW_DEG)
    pitch = math.radians(PITCH_DEG)
    distance = radius / math.sin(math.radians(FOV_DEG) / 2) / FILL
    eye = np.array([
        distance * math.cos(pitch) * math.sin(yaw),
        distance * math.sin(pitch),
        distance * math.cos(pitch) * math.cos(yaw),
    ])
    forward = -eye / np.linalg.norm(eye)
    up = np.array([0.0, 1.0, 0.0])
    right = np.cross(forward, up); right /= np.linalg.norm(right)
    true_up = np.cross(right, forward)
    m = np.eye(4)
    m[:3, 0] = right
    m[:3, 1] = true_up
    m[:3, 2] = -forward
    m[:3, 3] = eye
    return m

def is_blank(path):
    try:
        arr = np.array(Image.open(path).convert('RGB'))
    except Exception:
        return True
    return arr.std() < 2.0

def render_thumbnail(glb_path, out_path):
    tm = trimesh.load(glb_path, force='scene')
    if isinstance(tm, trimesh.Trimesh):
        tm = trimesh.Scene(tm)

    bbox_min, bbox_max = tm.bounds
    center = (bbox_min + bbox_max) / 2.0
    radius = float(np.linalg.norm(bbox_max - bbox_min) / 2.0)
    if radius == 0:
        radius = 1.0
    for _, geom in tm.geometry.items():
        geom.apply_translation(-center)

    tm.camera.fov = (FOV_DEG, FOV_DEG)
    tm.camera.resolution = (TH_SIZE, TH_SIZE)
    tm.camera_transform = camera_transform(radius)

    # visible=True is required on Windows. The offscreen path uses pyglet's
    # hidden-window code which returns a blank frame here. visible=True
    # briefly flashes a small window per file but produces a real render.
    png = tm.save_image(resolution=(TH_SIZE, TH_SIZE), visible=True, background=PAPER)
    img = Image.open(io.BytesIO(png)).convert('RGBA')
    img.save(out_path, format='WEBP', quality=88, method=6)

# Sweep blank artifacts from a prior run so the loop actually retries them.
swept = 0
for path in THUMBS.glob('*.webp'):
    if is_blank(path):
        path.unlink()
        swept += 1
print(f'swept {swept} blank thumbnails')

failures = []
for entry in entries:
    stem = entry['stem']
    glb = MODELS / f"{stem}_low.glb"
    out = THUMBS / f"{stem}.webp"
    if not glb.exists():
        failures.append((stem, 'missing glb'))
        continue
    if out.exists() and not is_blank(out):
        continue
    try:
        render_thumbnail(glb, out)
    except Exception as ex:
        failures.append((stem, repr(ex)))

valid = sum(
    1 for entry in entries
    if (THUMBS / f"{entry['stem']}.webp").exists()
    and not is_blank(THUMBS / f"{entry['stem']}.webp")
)
print(f'thumbnails: {valid} valid of {len(entries)}, {len(failures)} failures')
for stem, why in failures[:10]:
    print(' ', stem, '->', why)

swept 0 blank thumbnails
thumbnails: 60 valid of 60, 0 failures


## 3. Summary stats

In [4]:
def stats_for(values):
    if not values:
        return None
    return {
        'mean':   float(statistics.mean(values)),
        'median': float(statistics.median(values)),
        'stdev':  float(statistics.stdev(values)) if len(values) > 1 else 0.0,
        'min':    float(min(values)),
        'max':    float(max(values)),
    }

seconds = [e['seconds'] for e in entries if 'seconds' in e]
bytes_  = [e['glb_bytes'] for e in entries if 'glb_bytes' in e]
verts   = [e['vertex_count'] for e in entries if 'vertex_count' in e and e['vertex_count'] > 0]

summary = {
    'count': len(entries),
    'seconds':   stats_for(seconds),
    'glb_bytes': stats_for(bytes_),
    'vertex_count': stats_for(verts),
}
with open(STATS, 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))

{
  "count": 60,
  "seconds": {
    "mean": 5.78875,
    "median": 5.648999999999999,
    "stdev": 0.7932379604282733,
    "min": 4.944,
    "max": 11.303
  },
  "glb_bytes": {
    "mean": 455768.13333333336,
    "median": 449474.0,
    "stdev": 26626.36656250537,
    "min": 424484.0,
    "max": 539972.0
  },
  "vertex_count": {
    "mean": 10000.0,
    "median": 10000.0,
    "stdev": 0.0,
    "min": 10000.0,
    "max": 10000.0
  }
}


## 4. Validate every GLB

In [5]:
from pygltflib import GLTF2

broken = []
for entry in entries:
    stem = entry['stem']
    glb = MODELS / f"{stem}_low.glb"
    if not glb.exists():
        broken.append((stem, 'missing'))
        continue
    try:
        GLTF2().load(str(glb))
    except Exception as ex:
        broken.append((stem, repr(ex)))

print(f'{len(entries) - len(broken)} ok, {len(broken)} broken')
for stem, why in broken:
    print(' ', stem, '->', why)

60 ok, 0 broken


## 5. Optional: batch compress (off by default)

Requires `gltf-transform` CLI on PATH. Writes compressed copies to `public/models/compressed/`. Leave the cell un-run unless you need it.

In [6]:
RUN_COMPRESS = False
if RUN_COMPRESS:
    out_dir = MODELS / 'compressed'
    out_dir.mkdir(exist_ok=True)
    for entry in entries:
        stem = entry['stem']
        src = MODELS / f"{stem}_low.glb"
        dst = out_dir / f"{stem}_low.glb"
        if dst.exists() or not src.exists():
            continue
        subprocess.run(['gltf-transform', 'optimize', str(src), str(dst), '--compress', 'draco', '--texture-compress', 'webp'], check=True)
else:
    print('skipped (set RUN_COMPRESS = True to enable)')

skipped (set RUN_COMPRESS = True to enable)
